# Before Building Model

In [1]:
from IPython.display import clear_output
!pip3 install pytorch-tabnet
clear_output()

import pandas as pd
import numpy as np
import os
import random
import warnings
warnings.simplefilter('ignore')

from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetRegressor
import torch

TRAIN_PATH = "../input/house-prices-advanced-regression-techniques/train.csv"
TEST_PATH = "../input/house-prices-advanced-regression-techniques/test.csv"
SAMPLE_SUBMISSION_PATH = "../input/house-prices-advanced-regression-techniques/sample_submission.csv"
SUBMISSION_PATH = "submission.csv"

ID = "Id"
TARGET = "SalePrice"

SEED = 2022
def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything()

VAL_SIZE = 0.25
MAX_EPOCHS = 100
MODEL_PATIENCE = 500
MODEL_EVAL_MERIC = 'rmse'
MODEL_BATCH_SIZE = 1024
MODEL_VIRTUAL_BATCH_SIZE = 128


train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

def toLabelEncode(TRAIN_PATH,TEST_PATH):
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    
    cat_col = train.describe(include="O").columns.tolist()
    
    for col in cat_col:
        train[col] = train[col].astype("category").cat.codes
        test[col] = test[col].astype("category").cat.codes
        
    return train,test

train,test = toLabelEncode(TRAIN_PATH,TEST_PATH)

def checkNull_fillData(df):
    for col in df.columns:
        if len(df.loc[df[col].isnull() == True]) != 0:
            if df[col].dtype == "float64" or df[col].dtype == "int64":
                df.loc[df[col].isnull() == True,col] = df[col].mean()
            else:
                df.loc[df[col].isnull() == True,col] = df[col].mode()[0]
                
checkNull_fillData(train)
checkNull_fillData(test)

# Build Model

In [2]:
features = test.columns

X = train.drop([ID,TARGET],axis=1)
y= train[TARGET]
X_test = test.drop([ID],axis=1)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=VAL_SIZE) 

model = TabNetRegressor()
model.fit(
    X_train.values, y_train.values.reshape(-1, 1),
    eval_set=[(X_valid.values, y_valid.values.reshape(-1, 1))],
    eval_metric=[MODEL_EVAL_MERIC],
    max_epochs=MAX_EPOCHS,
    patience=MODEL_PATIENCE,
    batch_size=MODEL_BATCH_SIZE, 
    virtual_batch_size=MODEL_VIRTUAL_BATCH_SIZE,
    num_workers=0,
    drop_last=False
)

epoch 0  | loss: 38873818804.95342| val_0_rmse: 198786.63964|  0:00:00s
epoch 1  | loss: 38873496661.09954| val_0_rmse: 198817.3218|  0:00:00s
epoch 2  | loss: 38873237995.89406| val_0_rmse: 198670.3417|  0:00:00s
epoch 3  | loss: 38872976450.39634| val_0_rmse: 198373.43789|  0:00:00s
epoch 4  | loss: 38872716490.92968| val_0_rmse: 198706.68632|  0:00:01s
epoch 5  | loss: 38872447613.77899| val_0_rmse: 198576.38691|  0:00:01s
epoch 6  | loss: 38872139190.58995| val_0_rmse: 198116.68314|  0:00:01s
epoch 7  | loss: 38871813829.7863| val_0_rmse: 197605.58339|  0:00:01s
epoch 8  | loss: 38871493114.85662| val_0_rmse: 197709.09719|  0:00:01s
epoch 9  | loss: 38871143413.71324| val_0_rmse: 197932.63914|  0:00:01s
epoch 10 | loss: 38870670286.43653| val_0_rmse: 198277.46368|  0:00:01s
epoch 11 | loss: 38870231305.58539| val_0_rmse: 198789.68928|  0:00:01s
epoch 12 | loss: 38869760065.46119| val_0_rmse: 198797.21706|  0:00:02s
epoch 13 | loss: 38869194013.22374| val_0_rmse: 198801.92466|  0:00

# After Builiding Model

In [3]:
sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
sub[TARGET] = model.predict(X_test.values)
sub.to_csv(SUBMISSION_PATH,index=False)
sub.head()

,Id,SalePrice
0,1461,2642.218018
1,1462,18.833445
2,1463,504.284424
3,1464,502.632019
4,1465,14.397677
